# POCD-KansformerEPI v6 — Vertex AI Deployment

**Key v6 changes (vs v5):**
- **Enh/prom index extraction**: Extracts transformer features at exact enhancer/promoter positions (was generic pooling)
- **Positional encoding channel**: sym_log distance to nearest enh/prom boundary added as input channel (9 input channels, was 8)
- **500 epi tokens**: Single CNN layer + MaxPool1d(10) → 500 epi tokens (was 2-layer CNN → 128)
- **Epi BiLSTM**: Added 2-layer bidirectional LSTM to epi branch (matches reference)
- **720-dim FC head**: [enh_feat, prom_feat, attn_mean, attn_max] = 4×180 (was 2×180=360)
- **628 total tokens**: 128 seq + 500 epi (was 256)
- **Scheduler disabled**: ReduceLROnPlateau instead of CosineAnnealingWarmRestarts (was causing instability)
- **300 epochs**: Up from 100
- **weight_decay=0**: Matches reference (was 1e-8)

**Architecture**: hidden_dim=180, heads=6, depth=3, kan_hidden=64, dual-branch (POCD-ND seq + epi)

Run all cells in order. Training is launched from terminal (not notebook).


In [ ]:
# Stop any previous training and navigate to project
!pkill -f train.py 2>/dev/null || true
%cd /home/jupyter/POCD-KansformerEPI


In [ ]:
import os, glob

bengi = './data/BENGI'
feats = './data/genomic_data/CTCF_DNase_6histone_local.500.json'
ref = '/home/jupyter/POCD-KansformerEPI/data/reference/hg19.fa'

print('=== Data Diagnostics ===')
print(f'BENGI dir exists: {os.path.isdir(bengi)}')
if os.path.isdir(bengi):
    files = sorted(glob.glob(os.path.join(bengi, '*.tsv*')))
    print(f'  TSV files: {len(files)}')
    for f in files:
        sz = os.path.getsize(f) / 1e6
        print(f'    {os.path.basename(f)} ({sz:.1f} MB)')

print(f'\nFeats config exists: {os.path.isfile(feats)}')
print(f'Reference genome exists: {os.path.isfile(ref)}')

# Check genomic features
gd = './data/genomic_data'
if os.path.isdir(gd):
    jsons = [f for f in os.listdir(gd) if f.endswith('.json')]
    print(f'\nJSON configs in genomic_data: {len(jsons)}')
    for j in sorted(jsons):
        print(f'  {j}')

# Check processed .pt files (pre-processed epigenetic signals)
proc = './data/genomic_data/processed'
if os.path.isdir(proc):
    pts = sorted([f for f in os.listdir(proc) if f.endswith('.pt')])
    print(f'\nProcessed .pt files: {len(pts)}')
    for p in pts:
        sz = os.path.getsize(os.path.join(proc, p)) / 1e6
        print(f'  {p} ({sz:.1f} MB)')
else:
    print(f'\nProcessed dir not found at {proc}')


In [ ]:
!pip install pysam -q


## Write v6 Source Files

The following cells write all source files using `%%writefile`. Run them all.

In [ ]:
%%writefile configs/config.yaml
experiment_name: "POCD_Kansformer_v6"

paths:
  save_dir: "./checkpoints"
  bengi_dir: "./data/BENGI"
  feats_config: "./data/genomic_data/CTCF_DNase_6histone_local.500.json"
  ref_genome: "/home/jupyter/POCD-KansformerEPI/data/reference/hg19.fa"

data:
  sequence_length: 6000   # 3000bp enhancer + 3000bp promoter concatenated
  epigenetic_bins: 5000   # 2.5Mbp window / 500bp = 5000 bins
  n_epigenetic_features: 9  # 1 pos_enc + CTCF + DNase + 6 histone marks
  kmer_size: 3
  batch_size: 64            # Reduced from 128 — 628 tokens through KAN OOMs on L4 at 128
  bin_size: 500
  seq_len_bp: 2500000
  enhancer_window: 3000
  promoter_window: 3000
  encoder_fit_samples: 5000
  feats_order:
    - "CTCF"
    - "DNase"
    - "H3K27ac"
    - "H3K27me3"
    - "H3K36me3"
    - "H3K4me1"
    - "H3K4me3"
    - "H3K9me3"

model:
  hidden_dim: 180          # Match reference KansformerEPI
  num_heads: 6             # 180/6 = 30 per head
  num_layers: 3            # Match reference depth=3
  n_tokens: 128            # Sequence branch tokens (epi uses MaxPool1d(10)→500)
  dropout: 0.1             # Match reference
  head_dropout: 0.2        # FC head dropout
  kan_hidden: 64           # KAN FFN hidden dim [180, 64, 180]
  sa_da: 64
  sa_r: 32
  att_penalty: 0.1

training:
  lr: 0.0001               # 1e-4 (matches reference)
  epochs: 300              # Match reference (was 100)
  lambda_dist: 0.1
  grad_clip: 1.0
  patience: 10             # Match reference patience=10
  num_workers: 8
  weight_decay: 0          # Match reference (was 1e-8)
  use_cosine_scheduler: false  # Disabled — reference default (was true)
  att_penalty_coeff: 0.1
  valid_chroms:
    - "chr11"
    - "chr17"
  test_chroms:
    - "chr1"
    - "chr2"

augmentation:
  enabled: true
  rc_prob: 0.5
  epi_noise_std: 0.05
  shift_max_bins: 3

In [ ]:
%%writefile src/__init__.py



In [ ]:
%%writefile src/model_layers.py
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


class KANLinear(nn.Module):
    """
    Kolmogorov-Arnold Network Layer with B-spline basis functions.

    Computes: output = W_base * SiLU(x) + W_spline * B-splines(x)

    Matches the reference KansformerEPI implementation (grid_size=5, spline_order=3).
    """
    def __init__(
        self,
        in_features,
        out_features,
        grid_size=5,
        spline_order=3,
        scale_noise=0.1,
        scale_base=1.0,
        scale_spline=1.0,
        enable_standalone_scale_spline=True,
        base_activation=nn.SiLU,
        grid_eps=0.02,
        grid_range=(-1, 1),
    ):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.grid_size = grid_size
        self.spline_order = spline_order

        # Build uniform B-spline knot grid
        h = (grid_range[1] - grid_range[0]) / grid_size
        grid = (
            (torch.arange(-spline_order, grid_size + spline_order + 1) * h + grid_range[0])
            .expand(in_features, -1)
            .contiguous()
        )
        self.register_buffer("grid", grid)  # (in_features, grid_size + 2*spline_order + 1)

        # Parameters
        self.base_weight = nn.Parameter(torch.Tensor(out_features, in_features))
        self.spline_weight = nn.Parameter(
            torch.Tensor(out_features, in_features, grid_size + spline_order)
        )
        if enable_standalone_scale_spline:
            self.spline_scaler = nn.Parameter(torch.Tensor(out_features, in_features))
        else:
            self.spline_scaler = None

        self.scale_noise = scale_noise
        self.scale_base = scale_base
        self.scale_spline = scale_spline
        self.enable_standalone_scale_spline = enable_standalone_scale_spline
        self.base_activation = base_activation()
        self.grid_eps = grid_eps

        self.reset_parameters()

    def reset_parameters(self):
        nn.init.kaiming_uniform_(self.base_weight, a=math.sqrt(5) * self.scale_base)
        with torch.no_grad():
            noise = (
                (torch.rand(self.grid_size + 1, self.in_features, self.out_features) - 0.5)
                * self.scale_noise / self.grid_size
            )
            self.spline_weight.data.copy_(
                (self.scale_spline if not self.enable_standalone_scale_spline else 1.0)
                * self.curve2coeff(
                    self.grid.T[self.spline_order: -self.spline_order],
                    noise,
                )
            )
            if self.enable_standalone_scale_spline:
                nn.init.kaiming_uniform_(
                    self.spline_scaler, a=math.sqrt(5) * self.scale_spline
                )

    def b_splines(self, x):
        """Compute B-spline bases. x: (batch, in_features) -> (batch, in_features, n_bases)"""
        assert x.dim() == 2 and x.size(1) == self.in_features
        grid = self.grid  # (in_features, n_knots)
        x = x.unsqueeze(-1)
        bases = ((x >= grid[:, :-1]) & (x < grid[:, 1:])).to(x.dtype)
        for k in range(1, self.spline_order + 1):
            bases = (
                (x - grid[:, :-(k + 1)]) / (grid[:, k:-1] - grid[:, :-(k + 1)]) * bases[:, :, :-1]
            ) + (
                (grid[:, k + 1:] - x) / (grid[:, k + 1:] - grid[:, 1:(-k)]) * bases[:, :, 1:]
            )
        return bases.contiguous()

    def curve2coeff(self, x, y):
        """Fit B-spline coefficients via least squares. Used during initialization."""
        assert x.dim() == 2 and x.size(1) == self.in_features
        A = self.b_splines(x).transpose(0, 1)  # (in, batch, n_bases)
        B = y.transpose(0, 1)                   # (in, batch, out)
        solution = torch.linalg.lstsq(A, B).solution  # (in, n_bases, out)
        result = solution.permute(2, 0, 1)             # (out, in, n_bases)
        return result.contiguous()

    @property
    def scaled_spline_weight(self):
        if self.enable_standalone_scale_spline:
            return self.spline_weight * self.spline_scaler.unsqueeze(-1)
        return self.spline_weight

    def forward(self, x):
        assert x.dim() == 2 and x.size(1) == self.in_features
        # Base path: SiLU(x) @ W_base^T
        base_output = F.linear(self.base_activation(x), self.base_weight)
        # Spline path: B-splines(x) @ W_spline^T
        spline_output = F.linear(
            self.b_splines(x).view(x.size(0), -1),
            self.scaled_spline_weight.view(self.out_features, -1),
        )
        return base_output + spline_output

    @torch.no_grad()
    def update_grid(self, x, margin=0.01):
        """Adaptively update grid based on input data distribution."""
        assert x.dim() == 2 and x.size(1) == self.in_features
        batch = x.size(0)

        splines = self.b_splines(x).permute(1, 0, 2)
        orig_coeff = self.scaled_spline_weight.permute(1, 2, 0)
        unreduced_spline_output = torch.bmm(splines, orig_coeff).permute(1, 0, 2)

        x_sorted = torch.sort(x, dim=0)[0]
        grid_adaptive = x_sorted[
            torch.linspace(0, batch - 1, self.grid_size + 1, dtype=torch.int64, device=x.device)
        ]
        uniform_step = (x_sorted[-1] - x_sorted[0] + 2 * margin) / self.grid_size
        grid_uniform = (
            torch.arange(self.grid_size + 1, dtype=torch.float32, device=x.device).unsqueeze(1)
            * uniform_step + x_sorted[0] - margin
        )
        grid = self.grid_eps * grid_uniform + (1 - self.grid_eps) * grid_adaptive
        grid = torch.cat(
            [
                grid[:1] - uniform_step * torch.arange(self.spline_order, 0, -1, device=x.device).unsqueeze(1),
                grid,
                grid[-1:] + uniform_step * torch.arange(1, self.spline_order + 1, device=x.device).unsqueeze(1),
            ],
            dim=0,
        )
        self.grid.copy_(grid.T)
        self.spline_weight.data.copy_(self.curve2coeff(x, unreduced_spline_output))

    def regularization_loss(self, regularize_activation=1.0, regularize_entropy=1.0):
        """L1 + entropy regularization on spline weights."""
        l1_fake = self.spline_weight.abs().mean(-1)
        reg_activation = l1_fake.sum()
        p = l1_fake / reg_activation
        reg_entropy = -torch.sum(p * p.log())
        return regularize_activation * reg_activation + regularize_entropy * reg_entropy


class KAN(nn.Module):
    """Multi-layer KAN network. Chains KANLinear layers sequentially."""
    def __init__(
        self,
        layers_hidden,
        grid_size=5,
        spline_order=3,
        scale_noise=0.1,
        scale_base=1.0,
        scale_spline=1.0,
        base_activation=nn.SiLU,
        grid_eps=0.02,
        grid_range=(-1, 1),
    ):
        super().__init__()
        self.grid_size = grid_size
        self.spline_order = spline_order
        self.layers = nn.ModuleList()
        for in_f, out_f in zip(layers_hidden, layers_hidden[1:]):
            self.layers.append(
                KANLinear(
                    in_f, out_f,
                    grid_size=grid_size, spline_order=spline_order,
                    scale_noise=scale_noise, scale_base=scale_base,
                    scale_spline=scale_spline, base_activation=base_activation,
                    grid_eps=grid_eps, grid_range=grid_range,
                )
            )

    def forward(self, x, update_grid=False):
        for layer in self.layers:
            if update_grid:
                layer.update_grid(x)
            x = layer(x)
        return x

    def regularization_loss(self, regularize_activation=1.0, regularize_entropy=1.0):
        return sum(
            layer.regularization_loss(regularize_activation, regularize_entropy)
            for layer in self.layers
        )

In [ ]:
%%writefile src/encoding.py
import torch
import numpy as np
import itertools

class POCD_ND_Encoder:
    """
    Implements Position-Aware Oligonucleotide Composition Density with Negative Density (POCD-ND).
    Ref: Section 3.2.2 of Project Design.
    """
    def __init__(self, k=3):
        self.k = k
        self.kmers = [''.join(p) for p in itertools.product('ACGT', repeat=k)]
        self.kmer_to_idx = {kmer: i for i, kmer in enumerate(self.kmers)}
        self.num_kmers = len(self.kmers)
        self.pos_density_matrix = None
        self.neg_density_matrix = None

    def _compute_freq(self, sequences, seq_len):
        num_positions = seq_len - self.k + 1
        freq_matrix = np.zeros((self.num_kmers, num_positions))
        
        for seq in sequences:
            # Simple padding handling if seq is short
            loop_len = min(len(seq), seq_len) - self.k + 1
            for i in range(loop_len):
                kmer = seq[i : i + self.k]
                if kmer in self.kmer_to_idx:
                    freq_matrix[self.kmer_to_idx[kmer], i] += 1
        return freq_matrix

    def fit(self, pos_sequences, neg_sequences, seq_len):
        """Calculates global density matrices A^pos and A^neg."""
        print("Fitting POCD-ND Encoder...")
        A_pos = self._compute_freq(pos_sequences, seq_len)
        A_neg = self._compute_freq(neg_sequences, seq_len)
        
        epsilon = 1e-9
        # Normalize densities (Step 3)
        self.pos_density_matrix = (A_pos / (len(pos_sequences) + epsilon)) + epsilon
        self.neg_density_matrix = (A_neg / (len(neg_sequences) + epsilon)) + epsilon

    def transform(self, sequence):
        """Encodes a single sequence using Eq: Ratio * Min(Densities)."""
        seq_len = len(sequence)
        num_positions = seq_len - self.k + 1
        
        # Calculate global POCD map
        ratio = self.pos_density_matrix[:, :num_positions] / self.neg_density_matrix[:, :num_positions]
        min_den = np.minimum(self.pos_density_matrix[:, :num_positions], self.neg_density_matrix[:, :num_positions])
        global_map = ratio * min_den
        
        # Mask: Only activate k-mers present in the sequence
        encoded = np.zeros_like(global_map)
        for i in range(num_positions):
            kmer = sequence[i : i + self.k]
            if kmer in self.kmer_to_idx:
                idx = self.kmer_to_idx[kmer]
                encoded[idx, i] = global_map[idx, i]
                
        return torch.FloatTensor(encoded)

In [ ]:
%%writefile src/epi_data_pipeline.py
"""
Epigenetic signal data loading for POCD-KansformerEPI.

Reads the BENGI benchmark TSV files and pre-processed .pt epigenetic signal files
(same format as the original Kansformer), then adds DNA sequence extraction for
the POCD-ND encoding branch.

Data layout expected:
  data/
    BENGI/
      GM12878.HiC-Benchmark.v3.tsv.gz   (or .tsv)
      ...
    genomic_data/
      processed/
        bigWig_GM12878_DNase.500bp.pt
        narrowPeak_GM12878_CTCF.500bp.pt
        ...
      CTCF_DNase_6histone_local.500.json  (feature config)
"""

import os
import sys
import csv
import gzip
import json
import math
import random
import numpy as np
from collections import OrderedDict
from typing import Dict, List, Optional, Tuple, Union

import torch
from torch.utils.data import Dataset

# ---------------------------------------------------------------------------
# hg19 chromosome sizes (for boundary checking)
# ---------------------------------------------------------------------------
HG19_CHROMSIZE = {
    "chr1": 249250621, "chr2": 243199373, "chr3": 198022430,
    "chr4": 191154276, "chr5": 180915260, "chr6": 171115067,
    "chr7": 159138663, "chr8": 146364022, "chr9": 141213431,
    "chr10": 135534747, "chr11": 135006516, "chr12": 133851895,
    "chr13": 115169878, "chr14": 107349540, "chr15": 102531392,
    "chr16": 90354753, "chr17": 81195210, "chr18": 78077248,
    "chr19": 59128983, "chr20": 63025520, "chr21": 48129895,
    "chr22": 51304566, "chrX": 155270560, "chrY": 59373566,
    "chrM": 16569, "chrMT": 16569,
}


def _open_file(fn: str):
    """Open plain text or gzipped file."""
    if fn.endswith(".gz"):
        return gzip.open(fn, "rt")
    return open(fn, "rt")


def _normalize_minmax(tensor: torch.Tensor) -> torch.Tensor:
    """Per-feature (row) min-max normalisation."""
    mn = tensor.min(dim=1, keepdim=True)[0]
    mx = tensor.max(dim=1, keepdim=True)[0]
    return (tensor - mn) / (mx - mn + 1e-8)


def _sym_log(x: torch.Tensor) -> torch.Tensor:
    """Symmetric log: sign(x) * log10(1 + |x|).  Matches reference epi_dataset.py."""
    return torch.sign(x) * torch.log10(1 + torch.abs(x))


# ===================================================================
# Core data-loading class
# ===================================================================
class EPIGenomicDataset(Dataset):
    """
    PyTorch Dataset for Enhancer-Promoter Interaction prediction.

    Each sample returns:
        epi_features : Tensor of shape (num_feats, num_bins)
        enhancer_seq : str   (DNA sequence around enhancer center)
        promoter_seq : str   (DNA sequence around promoter center)
        distance     : float (log-scaled genomic distance)
        label        : float (0 or 1)
    """

    _FEATS_ORDER_8 = [
        "CTCF", "DNase", "H3K27ac", "H3K27me3",
        "H3K36me3", "H3K4me1", "H3K4me3", "H3K9me3",
    ]
    _FEATS_ORDER_6 = [
        "CTCF", "DNase", "H3K27ac", "H3K4me1", "H3K4me3", "H3K9ac",
    ]

    def __init__(
        self,
        bengi_paths: Union[str, List[str]],
        feats_config_path: str,
        feats_order: Optional[List[str]] = None,
        seq_len: int = 2_500_000,
        bin_size: int = 500,
        enhancer_window: int = 3000,
        promoter_window: int = 3000,
        ref_genome_path: Optional[str] = None,
        normalize_epi: bool = True,
    ):
        """
        Parameters
        ----------
        bengi_paths : path(s) to BENGI TSV / TSV.gz benchmark files
        feats_config_path : path to the JSON mapping cell→mark→.pt file
        feats_order : ordered list of epigenetic mark names to use
        seq_len : genomic window size (bp) for epigenetic features
        bin_size : bin size (bp) for the .pt signal files
        enhancer_window : bp length of DNA to extract around enhancer center
        promoter_window : bp length of DNA to extract around promoter center
        ref_genome_path : path to hg19 FASTA (indexed). If None, returns
                          'N'-padded dummy sequences.
        normalize_epi : apply per-feature min-max normalisation
        """
        super().__init__()

        if isinstance(bengi_paths, str):
            bengi_paths = [bengi_paths]
        self.bengi_paths = bengi_paths

        self.seq_len = int(seq_len)
        self.bin_size = int(bin_size)
        assert self.seq_len % self.bin_size == 0
        self.num_bins = self.seq_len // self.bin_size

        self.enhancer_window = enhancer_window
        self.promoter_window = promoter_window
        self.normalize_epi = normalize_epi

        # Determine feature order
        if feats_order is None:
            feats_order = list(self._FEATS_ORDER_8)
        self.feats_order = list(feats_order)
        self.num_feats = len(self.feats_order)

        # Load feature config JSON
        with open(feats_config_path, "r") as f:
            self.feats_config = json.load(f)
        base_location = self.feats_config.pop("_location", None)
        if base_location is None:
            base_location = os.path.dirname(os.path.abspath(feats_config_path))
        # Resolve absolute paths for each .pt file
        for cell, assays in list(self.feats_config.items()):
            if not isinstance(assays, dict):
                # Skip non-dict entries (e.g. metadata keys)
                del self.feats_config[cell]
                continue
            for mark, fn in assays.items():
                if not isinstance(fn, str):
                    continue
                if not os.path.isabs(fn):
                    assays[mark] = os.path.join(base_location, fn)

        # Chromosome → max bins
        self.chrom_bins = {
            ch: (sz // bin_size)
            for ch, sz in HG19_CHROMSIZE.items()
        }

        # Reference genome (optional)
        self.ref_genome = None
        if ref_genome_path and os.path.exists(ref_genome_path):
            try:
                import pysam
                self.ref_genome = pysam.FastaFile(ref_genome_path)
                print(f"  Reference genome loaded: {ref_genome_path}")
            except ImportError:
                print("  WARNING: pysam not installed – using dummy sequences")

        # Storage
        self.samples: List[dict] = []
        self.feats: Dict[str, Dict[str, dict]] = {}  # cell → mark → {chrom: tensor}

        self._load_datasets()
        print(f"  EPIGenomicDataset ready: {len(self.samples)} samples, "
              f"{self.num_feats} marks, {self.num_bins} bins")

    # ---------------------------------------------------------------
    def _load_datasets(self):
        for fn in self.bengi_paths:
            with _open_file(fn) as f:
                for line in f:
                    fields = [x for x in line.strip().split("\t") if x]
                    if len(fields) < 10:
                        continue
                    (label, dist, chrom,
                     enh_start, enh_end, enh_name,
                     _prom_chrom,
                     prom_start, prom_end, prom_name) = fields[:10]

                    cell = enh_name.split("|")[1]

                    enh_coord = (int(enh_start) + int(enh_end)) // 2
                    p_coords = prom_name.split("|")[0].split(":")[-1].split("-")
                    tss_coord = (int(p_coords[0]) + int(p_coords[1])) // 2

                    mid = (enh_coord + tss_coord) // 2
                    seq_begin = mid - self.seq_len // 2
                    seq_end = mid + self.seq_len // 2

                    enh_bin = enh_coord // self.bin_size
                    prom_bin = tss_coord // self.bin_size
                    start_bin = seq_begin // self.bin_size
                    stop_bin = seq_end // self.bin_size

                    left_pad, right_pad = 0, 0
                    if start_bin < 0:
                        left_pad = abs(start_bin)
                        start_bin = 0
                    if chrom in self.chrom_bins and stop_bin > self.chrom_bins[chrom]:
                        right_pad = stop_bin - self.chrom_bins[chrom]
                        stop_bin = self.chrom_bins[chrom]

                    self.samples.append({
                        "start_bin": start_bin,
                        "stop_bin": stop_bin,
                        "left_pad": left_pad,
                        "right_pad": right_pad,
                        "enh_bin": enh_bin,
                        "prom_bin": prom_bin,
                        "enh_coord": enh_coord,
                        "prom_coord": tss_coord,
                        "cell": cell,
                        "chrom": chrom,
                        "label": int(label),
                        "dist": float(dist),
                    })

                    # Lazy-load .pt files per cell line
                    if cell not in self.feats:
                        self._load_cell_features(cell)

    def _load_cell_features(self, cell: str):
        """Load all epigenetic signal .pt files for a cell line."""
        if cell not in self.feats_config:
            print(f"  WARNING: cell '{cell}' not in feats_config – skipping")
            self.feats[cell] = {}
            return
        self.feats[cell] = {}
        for mark in self.feats_order:
            if mark not in self.feats_config[cell]:
                print(f"  WARNING: mark '{mark}' not in config for {cell}")
                continue
            pt_path = self.feats_config[cell][mark]
            if not os.path.exists(pt_path):
                print(f"  WARNING: {pt_path} not found")
                continue
            self.feats[cell][mark] = torch.load(pt_path, map_location="cpu", weights_only=False)
        print(f"  Loaded {len(self.feats[cell])}/{self.num_feats} marks for {cell}")

    # ---------------------------------------------------------------
    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx) -> dict:
        s = self.samples[idx]
        start_bin = s["start_bin"]
        stop_bin = s["stop_bin"]
        left_pad = s["left_pad"]
        right_pad = s["right_pad"]
        cell = s["cell"]
        chrom = s["chrom"]

        # --- Epigenetic features: (num_feats, num_bins) ---
        feat_rows = []
        for mark in self.feats_order:
            if cell in self.feats and mark in self.feats[cell] and chrom in self.feats[cell][mark]:
                row = self.feats[cell][mark][chrom][start_bin:stop_bin]
            else:
                row = torch.zeros(stop_bin - start_bin)
            feat_rows.append(row.unsqueeze(0))  # (1, bins)

        ar = torch.cat(feat_rows, dim=0)  # (num_feats, stop-start)

        # Pad if near chromosome edges
        if left_pad > 0 or right_pad > 0:
            ar = torch.cat([
                torch.zeros(self.num_feats, left_pad),
                ar,
                torch.zeros(self.num_feats, right_pad),
            ], dim=1)

        if self.normalize_epi:
            ar = _normalize_minmax(ar)

        # --- Enhancer / Promoter relative indices (for index-based feature extraction) ---
        enh_idx = s["enh_bin"] - start_bin + left_pad
        prom_idx = s["prom_bin"] - start_bin + left_pad

        # Clamp to valid range
        enh_idx = max(0, min(enh_idx, self.num_bins - 1))
        prom_idx = max(0, min(prom_idx, self.num_bins - 1))

        # --- Positional encoding channel (matches reference epi_dataset.py) ---
        # V-shaped signal encoding distance to nearest enh/prom boundary
        pos = torch.arange(self.num_bins).float()
        d1 = pos - min(enh_idx, prom_idx)       # distance from left boundary
        d2 = max(enh_idx, prom_idx) - pos        # distance from right boundary
        pos_enc_raw = torch.stack([d1, d2], dim=0)   # (2, num_bins)
        pos_enc = _sym_log(pos_enc_raw.min(dim=0)[0]).unsqueeze(0)  # (1, num_bins)

        # Prepend positional encoding to features (NOT normalized)
        ar = torch.cat([pos_enc, ar], dim=0)     # (num_feats + 1, num_bins)

        # --- DNA sequences ---
        enh_seq = self._extract_seq(chrom, s["enh_coord"], self.enhancer_window)
        prom_seq = self._extract_seq(chrom, s["prom_coord"], self.promoter_window)

        # Distance: log-scaled (same formula as original Kansformer)
        dist_scaled = math.log2(1 + 500_000 / max(s["dist"], 1.0))

        return {
            "epi": ar,                                           # (num_feats+1, num_bins)
            "enhancer_seq": enh_seq,                             # str
            "promoter_seq": prom_seq,                            # str
            "label": torch.tensor([s["label"]], dtype=torch.float),
            "dist": torch.tensor([dist_scaled], dtype=torch.float),
            "enh_idx": torch.tensor([enh_idx], dtype=torch.float),
            "prom_idx": torch.tensor([prom_idx], dtype=torch.float),
        }

    def _extract_seq(self, chrom: str, center: int, window: int) -> str:
        """Extract a DNA sequence of `window` bp centred on `center`."""
        half = window // 2
        start = max(0, center - half)
        end = start + window
        if self.ref_genome is not None:
            try:
                seq = self.ref_genome.fetch(chrom, start, end).upper()
                if len(seq) < window:
                    seq = seq + "N" * (window - len(seq))
                return seq
            except Exception:
                pass
        # Fallback: N-padded dummy (replaced by real sequences on Vertex AI)
        return "N" * window

    # ---------------------------------------------------------------
    # Utility: split by chromosome for cross-validation
    # ---------------------------------------------------------------
    def get_chrom_groups(self) -> np.ndarray:
        """Return an array of chromosome strings for GroupKFold splitting."""
        return np.array([s["chrom"] for s in self.samples])

    def get_labels(self) -> np.ndarray:
        return np.array([s["label"] for s in self.samples])

    def get_distances(self) -> np.ndarray:
        return np.array([s["dist"] for s in self.samples])


# ===================================================================
# Alternative: Load from EPIPDLF CSV files (simpler pair format)
# ===================================================================
def load_epipdlf_csv(csv_path: str) -> List[dict]:
    """
    Read an EPIPDLF CSV file and return a list of dicts with keys:
        enhancer_chrom, enhancer_start, enhancer_end,
        promoter_chrom, promoter_start, promoter_end,
        label, distance
    """
    pairs = []
    with open(csv_path, "r") as f:
        reader = csv.DictReader(f)
        for row in reader:
            pairs.append({
                "enhancer_chrom": row["enhancer_chrom"],
                "enhancer_start": int(row["enhancer_start"]),
                "enhancer_end": int(row["enhancer_end"]),
                "promoter_chrom": row["promoter_chrom"],
                "promoter_start": int(row["promoter_start"]),
                "promoter_end": int(row["promoter_end"]),
                "label": int(row["label"]),
                "distance": int(row["enhancer_distance_to_promoter"]),
            })
    return pairs


# ===================================================================
# Quick self-test
# ===================================================================
if __name__ == "__main__":
    import argparse
    p = argparse.ArgumentParser()
    p.add_argument("--bengi", nargs="+", required=True)
    p.add_argument("--feats-config", required=True)
    p.add_argument("--feats-order", nargs="+", default=None)
    args = p.parse_args()

    ds = EPIGenomicDataset(
        bengi_paths=args.bengi,
        feats_config_path=args.feats_config,
        feats_order=args.feats_order,
    )
    sample = ds[0]
    print(f"epi shape: {sample['epi'].shape}")
    print(f"enhancer_seq length: {len(sample['enhancer_seq'])}")
    print(f"label: {sample['label']}, dist: {sample['dist']}")


In [ ]:
%%writefile src/dataset.py
"""
PyTorch Dataset wrapper that applies POCD-ND encoding on-the-fly.

Can be used in two modes:
  1. Wrapping an EPIGenomicDataset (real genomic data)
  2. Standalone with synthetic data (for pipeline testing)
"""

import torch
from torch.utils.data import Dataset
import numpy as np
# ---- Augmentation helpers ----
_COMP = str.maketrans('ACGTNacgtn', 'TGCANtgcan')

def _reverse_complement(seq: str) -> str:
    """Return reverse complement of a DNA string."""
    return seq.translate(_COMP)[::-1]

class EPIDataset(Dataset):
    """
    Wrapper that applies POCD-ND encoding to raw DNA sequences.

    Each __getitem__ returns:
        seq      : Tensor (64, L)   – POCD-ND encoded sequence
        epi      : Tensor (n_feats+1, n_bins) – pos_enc + epigenetic features
        label    : Tensor (1,)
        dist     : Tensor (1,)
        enh_idx  : Tensor (1,)   – enhancer bin index (in 5000-bin space)
        prom_idx : Tensor (1,)   – promoter bin index (in 5000-bin space)
    """

    def __init__(
        self,
        config: dict,
        encoder,
        source_dataset=None,
        sequences=None,
        epi_features=None,
        labels=None,
        distances=None,
        num_synthetic: int = 0,
        concat_enh_prom: bool = True,
    ):
        """
        Parameters
        ----------
        config : dict from config.yaml
        encoder : fitted POCD_ND_Encoder
        source_dataset : EPIGenomicDataset instance (preferred for real data)
        sequences : list of DNA strings (alternative manual input)
        epi_features : array (N, n_feats, n_bins) or list of arrays
        labels, distances : arrays / lists
        num_synthetic : generate N synthetic samples for testing
        concat_enh_prom : if True, concatenate enhancer+promoter sequences
                          from source_dataset into a single input string
        """
        self.seq_len = config["data"]["sequence_length"]
        self.bins = config["data"]["epigenetic_bins"]
        self.n_epi = config["data"]["n_epigenetic_features"]
        self.encoder = encoder
        self.concat_enh_prom = concat_enh_prom

        # Augmentation settings (toggled per-phase in training loop)
        self.augment = False
        aug_cfg = config.get("augmentation", {})
        self.aug_rc_prob = aug_cfg.get("rc_prob", 0.5)
        self.aug_epi_noise_std = aug_cfg.get("epi_noise_std", 0.05)
        self.aug_shift_max = aug_cfg.get("shift_max_bins", 3)

        self.mode = None
        self._source = None
        self._manual_data = []

        if source_dataset is not None:
            self.mode = "pipeline"
            self._source = source_dataset
            print(f"EPIDataset: wrapping {len(self._source)} pipeline samples")

        elif sequences is not None:
            self.mode = "manual"
            self._build_manual(sequences, epi_features, labels, distances)

        elif num_synthetic > 0:
            self.mode = "synthetic"
            self._build_synthetic(num_synthetic)

        else:
            raise ValueError(
                "Provide source_dataset, sequences, or num_synthetic > 0"
            )

    # ------------------------------------------------------------------
    def _build_manual(self, sequences, epi_features, labels, distances):
        n = len(sequences)
        if epi_features is None:
            epi_features = [
                np.zeros((self.n_epi, self.bins), dtype=np.float32)
                for _ in range(n)
            ]
        if labels is None:
            labels = [0.0] * n
        if distances is None:
            distances = [0.0] * n

        for i in range(n):
            seq = self._pad_or_trim(sequences[i])
            epi = self._ensure_epi_shape(epi_features[i])
            self._manual_data.append(
                (seq, epi, float(labels[i]), float(distances[i]))
            )
        print(f"EPIDataset: {n} manual samples loaded")

    def _build_synthetic(self, n):
        bases = list("ACGT")
        for _ in range(n):
            seq = "".join(np.random.choice(bases, size=self.seq_len))
            epi = np.random.rand(self.n_epi, self.bins).astype(np.float32)
            label = float(np.random.randint(0, 2))
            dist = float(np.random.rand() * 10.0)
            self._manual_data.append((seq, epi, label, dist))
        print(f"EPIDataset: {n} synthetic samples generated")

    # ------------------------------------------------------------------
    def _pad_or_trim(self, seq: str) -> str:
        if len(seq) > self.seq_len:
            return seq[: self.seq_len]
        if len(seq) < self.seq_len:
            return seq + "N" * (self.seq_len - len(seq))
        return seq

    def _ensure_epi_shape(self, epi):
        """Ensure epi is float32 with shape (n_epi, bins)."""
        epi = np.asarray(epi, dtype=np.float32)
        if epi.shape == (self.n_epi, self.bins):
            return epi
        if epi.ndim == 2 and epi.shape == (self.bins, self.n_epi):
            return epi.T
        out = np.zeros((self.n_epi, self.bins), dtype=np.float32)
        r = min(epi.shape[0], self.n_epi)
        c = min(epi.shape[1] if epi.ndim > 1 else self.bins, self.bins)
        if epi.ndim == 1:
            out[0, : min(len(epi), self.bins)] = epi[: self.bins]
        else:
            out[:r, :c] = epi[:r, :c]
        return out

    # ------------------------------------------------------------------
    def __len__(self):
        if self.mode == "pipeline":
            return len(self._source)
        return len(self._manual_data)

    def __getitem__(self, idx):
        if self.mode == "pipeline":
            return self._getitem_pipeline(idx)
        return self._getitem_manual(idx)

    def _getitem_pipeline(self, idx):
        """Retrieve from EPIGenomicDataset and apply POCD-ND encoding."""
        raw = self._source[idx]
        epi = raw["epi"].clone()  # (num_feats, num_bins) from pipeline

        # Build DNA input string
        if self.concat_enh_prom:
            dna = raw["enhancer_seq"] + raw["promoter_seq"]
        else:
            dna = raw["enhancer_seq"]
        dna = self._pad_or_trim(dna)

        # --- Data augmentation (training only) ---
        if self.augment:
            # 1. Reverse complement with probability rc_prob
            if np.random.random() < self.aug_rc_prob:
                dna = _reverse_complement(dna)

            # 2. Small Gaussian noise on epigenetic features
            if self.aug_epi_noise_std > 0:
                epi = epi + torch.randn_like(epi) * self.aug_epi_noise_std
                epi = epi.clamp(min=0)  # epigenetic signals are non-negative

            # 3. Random circular shift of epigenetic bins (simulates coord jitter)
            if self.aug_shift_max > 0:
                shift = np.random.randint(-self.aug_shift_max, self.aug_shift_max + 1)
                if shift != 0:
                    epi = torch.roll(epi, shifts=shift, dims=1)

        seq_enc = self.encoder.transform(dna)  # Tensor (64, L)

        return {
            "seq": seq_enc,
            "epi": epi.float(),
            "label": raw["label"],
            "dist": raw["dist"],
            "enh_idx": raw["enh_idx"],
            "prom_idx": raw["prom_idx"],
        }

    def _getitem_manual(self, idx):
        seq, epi, label, dist = self._manual_data[idx]
        seq_enc = self.encoder.transform(seq)  # Tensor (64, L)
        return {
            "seq": seq_enc,
            "epi": torch.tensor(epi, dtype=torch.float32),
            "label": torch.tensor([label], dtype=torch.float32),
            "dist": torch.tensor([dist], dtype=torch.float32),
            "enh_idx": torch.tensor([2500.0], dtype=torch.float32),  # default midpoint
            "prom_idx": torch.tensor([2500.0], dtype=torch.float32),
        }

In [ ]:
%%writefile src/model.py
import torch
import torch.nn as nn
import math
from src.model_layers import KANLinear, KAN


# ---------------------------------------------------------------------------
# DropPath (stochastic depth) — matches reference kanformer.py
# ---------------------------------------------------------------------------
def drop_path(x, drop_prob: float = 0.0, training: bool = False):
    if drop_prob == 0.0 or not training:
        return x
    keep_prob = 1 - drop_prob
    shape = (x.shape[0],) + (1,) * (x.ndim - 1)
    random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
    random_tensor.floor_()
    return x.div(keep_prob) * random_tensor


class DropPath(nn.Module):
    def __init__(self, drop_prob=None):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        return drop_path(x, self.drop_prob, self.training)


# ---------------------------------------------------------------------------
# Multi-Head Self-Attention — matches reference kanformer.py
# ---------------------------------------------------------------------------
class Attention(nn.Module):
    def __init__(self, dim, num_heads=8, qkv_bias=True, attn_drop=0.0, proj_drop=0.0):
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5
        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x


# ---------------------------------------------------------------------------
# KAN-Transformer Block — PRE-NORM with KAN replacing FFN
# Matches reference: x = x + DropPath(Attn(LN(x)))
#                    x = x + DropPath(KAN(LN(x).reshape(-1,d)).reshape(b,t,d))
# ---------------------------------------------------------------------------
class KANBlock(nn.Module):
    """Single Transformer encoder layer with KAN replacing FFN (pre-norm)."""
    def __init__(self, dim, num_heads, kan_hidden=64, qkv_bias=True,
                 drop=0.0, attn_drop=0.0, drop_path_rate=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, eps=1e-5)
        self.attn = Attention(dim, num_heads=num_heads, qkv_bias=qkv_bias,
                              attn_drop=attn_drop, proj_drop=drop)
        self.drop_path = DropPath(drop_path_rate) if drop_path_rate > 0.0 else nn.Identity()
        self.norm2 = nn.LayerNorm(dim, eps=1e-5)
        # KAN([dim, kan_hidden, dim]) replaces the FFN
        self.kan = KAN([dim, kan_hidden, dim])

    def forward(self, x):
        b, t, d = x.shape
        # Pre-norm attention
        x = x + self.drop_path(self.attn(self.norm1(x)))
        # Pre-norm KAN (must reshape to 2D for KANLinear)
        x = x + self.drop_path(
            self.kan(self.norm2(x).reshape(-1, d)).reshape(b, t, d)
        )
        return x


# ---------------------------------------------------------------------------
# KAN-Transformer Encoder — stacks KANBlocks with linearly increasing drop path
# ---------------------------------------------------------------------------
class KANTransformer(nn.Module):
    def __init__(self, embed_dim=180, depth=3, num_heads=6, kan_hidden=64,
                 qkv_bias=True, drop=0.1, attn_drop=0.0, drop_path_rate=0.0):
        super().__init__()
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, depth)]
        self.blocks = nn.Sequential(*[
            KANBlock(
                dim=embed_dim, num_heads=num_heads, kan_hidden=kan_hidden,
                qkv_bias=qkv_bias, drop=drop, attn_drop=attn_drop,
                drop_path_rate=dpr[i],
            )
            for i in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim, eps=1e-5)

    def forward(self, x):
        x = self.blocks(x)
        x = self.norm(x)
        return x


# ---------------------------------------------------------------------------
# Positional Encoding (sinusoidal, same as reference)
# ---------------------------------------------------------------------------
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


# ---------------------------------------------------------------------------
# Self-Attention Pooling (structured attention, Lin et al. 2017)
# ---------------------------------------------------------------------------
class SelfAttentionPooling(nn.Module):
    def __init__(self, d_model, da=64, r=32):
        super().__init__()
        self.r = r
        self.ws1 = nn.Linear(d_model, da, bias=False)
        self.ws2 = nn.Linear(da, r, bias=False)

    def forward(self, H):
        A = torch.softmax(self.ws2(torch.tanh(self.ws1(H))), dim=1)  # (B, S, r)
        A = A.transpose(1, 2)  # (B, r, S)
        M = torch.bmm(A, H)   # (B, r, d_model)
        return M, A

    def penalization_term(self, A):
        AAT = torch.bmm(A, A.transpose(1, 2))
        I = torch.eye(self.r, device=A.device).unsqueeze(0)
        return torch.norm(AAT - I, p='fro', dim=(1, 2)).mean()


# ---------------------------------------------------------------------------
# POCD-KansformerEPI v6 — Dual-branch with enh/prom index feature extraction
#
# Key changes vs v5:
#   1. Epi CNN: single layer + MaxPool1d(10) → 500 tokens (was 2-layer → 128)
#   2. Epi BiLSTM: added (matches reference)
#   3. Enh/prom index extraction from transformer output (like reference)
#   4. FC head input: 4*d_model=720 (was 2*d_model=360)
#   5. Total tokens: 128 seq + 500 epi = 628 (was 256)
# ---------------------------------------------------------------------------
class Kansformer(nn.Module):
    def __init__(self, config):
        super().__init__()
        d_model = config['model']['hidden_dim']       # 180
        n_seq_tokens = config['model'].get('n_tokens', 128)
        drop = config['model'].get('dropout', 0.1)
        depth = config['model'].get('num_layers', 3)
        num_heads = config['model'].get('num_heads', 6)
        kan_hidden = config['model'].get('kan_hidden', 64)

        self.n_seq_tokens = n_seq_tokens
        self.epi_pool_factor = 10  # MaxPool1d kernel → 5000/10 = 500 epi tokens
        self.n_epi_tokens = config['data']['epigenetic_bins'] // self.epi_pool_factor  # 500
        self.d_model = d_model

        # --- Sequence Branch (POCD-ND encoded: 64 channels x L) ---
        self.seq_cnn = nn.Sequential(
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(128, d_model, kernel_size=5, padding=2),
            nn.BatchNorm1d(d_model),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(n_seq_tokens),
        )

        # Seq BiLSTM (2 layers, bidirectional)
        self.seq_bilstm = nn.LSTM(
            d_model, d_model // 2, batch_first=True,
            bidirectional=True, num_layers=2, dropout=drop,
        )
        self.seq_drop = nn.Dropout(drop)

        # --- Epigenetic Branch (n_epi channels x 5000 bins) ---
        # Single CNN layer + MaxPool1d(10) → 500 tokens (matches reference)
        n_epi = config['data']['n_epigenetic_features']  # 9 (8 epi + 1 pos_enc)
        self.epi_cnn = nn.Sequential(
            nn.Conv1d(n_epi, d_model, kernel_size=11, padding=5),
            nn.BatchNorm1d(d_model),
            nn.LeakyReLU(),
            nn.MaxPool1d(self.epi_pool_factor),  # 5000 → 500 tokens
        )

        # Epi BiLSTM (matches reference)
        self.epi_bilstm = nn.LSTM(
            d_model, d_model // 2, batch_first=True,
            bidirectional=True, num_layers=2, dropout=drop,
        )
        self.epi_drop = nn.Dropout(drop)

        # --- Fusion + Positional Encoding ---
        total_tokens = n_seq_tokens + self.n_epi_tokens  # 128 + 500 = 628
        self.proj_drop = nn.Dropout(drop)
        self.pos_enc = PositionalEncoding(d_model, max_len=total_tokens + 2)

        # --- KAN-Transformer Encoder (KAN replaces FFN) ---
        self.transformer = KANTransformer(
            embed_dim=d_model, depth=depth, num_heads=num_heads,
            kan_hidden=kan_hidden, qkv_bias=True, drop=drop,
            attn_drop=0.0, drop_path_rate=0.0,
        )

        # --- Self-Attention Pooling ---
        sa_da = config['model'].get('sa_da', 64)
        sa_r = config['model'].get('sa_r', 32)
        self.att_pool = SelfAttentionPooling(d_model, da=sa_da, r=sa_r)

        # Pool: [enh_feat, prom_feat, attn_mean, attn_max] = 4 * d_model
        pool_dim = d_model * 4  # 720 (was 360)

        # --- Classification Head (matches reference structure) ---
        self.head_drop = nn.Dropout(config['model'].get('head_dropout', 0.2))
        self.fc_linear = nn.Linear(pool_dim, 128)
        self.fc_kan1 = KANLinear(128, 64)
        self.fc_kan2 = KANLinear(64, 1)

        # --- Distance Regression Head ---
        self.dist_kan1 = KANLinear(pool_dim, d_model)
        self.dist_kan2 = KANLinear(d_model, 1)

    def forward(self, seq, epi, enh_idx, prom_idx):
        B = seq.size(0)

        # --- Sequence branch ---
        x_seq = self.seq_cnn(seq)                # (B, d_model, n_seq_tokens)
        x_seq = x_seq.permute(0, 2, 1)          # (B, n_seq_tokens, d_model)
        x_seq, _ = self.seq_bilstm(x_seq)       # (B, n_seq_tokens, d_model)
        x_seq = self.seq_drop(x_seq)

        # --- Epigenetic branch ---
        x_epi = self.epi_cnn(epi)                # (B, d_model, n_epi_tokens)
        x_epi = x_epi.permute(0, 2, 1)          # (B, n_epi_tokens, d_model)
        x_epi, _ = self.epi_bilstm(x_epi)       # (B, n_epi_tokens, d_model)
        x_epi = self.epi_drop(x_epi)

        # --- Concat + positional encoding ---
        z = torch.cat([x_seq, x_epi], dim=1)     # (B, n_seq + n_epi, d_model)
        z = self.proj_drop(z)
        z = self.pos_enc(z)

        # --- KAN-Transformer ---
        z = self.transformer(z)                   # (B, n_seq + n_epi, d_model)

        # --- Self-Attention Pooling ---
        M, A = self.att_pool(z)                   # M: (B, r, d_model), A: (B, r, S)

        # --- Enhancer / Promoter index-based feature extraction ---
        # Map bin indices (5000-space) to token indices in the epi portion of z
        enh_token = self.n_seq_tokens + torch.div(
            enh_idx.long().view(B), self.epi_pool_factor, rounding_mode='trunc')
        prom_token = self.n_seq_tokens + torch.div(
            prom_idx.long().view(B), self.epi_pool_factor, rounding_mode='trunc')

        # Clamp to valid range
        max_idx = self.n_seq_tokens + self.n_epi_tokens - 1
        enh_token = enh_token.clamp(self.n_seq_tokens, max_idx)
        prom_token = prom_token.clamp(self.n_seq_tokens, max_idx)

        # Gather features at enh/prom positions (per-sample indexing)
        batch_idx = torch.arange(B, device=z.device)
        enh_feat = z[batch_idx, enh_token, :]     # (B, d_model)
        prom_feat = z[batch_idx, prom_token, :]   # (B, d_model)

        # Attention-weighted mean and max
        attn_mean = M.mean(dim=1)                 # (B, d_model)
        attn_max = M.max(dim=1)[0]                # (B, d_model)

        z_pool = torch.cat([enh_feat, prom_feat, attn_mean, attn_max], dim=1)  # (B, 4*d_model)

        # --- Classification head ---
        feats = self.head_drop(z_pool)
        feats = self.fc_linear(feats)             # (B, 128)
        feats = self.fc_kan1(feats)               # (B, 64)
        cls_out = self.fc_kan2(feats)             # (B, 1)

        # --- Distance head ---
        dist_feats = self.head_drop(z_pool)
        dist_feats = self.dist_kan1(dist_feats)   # (B, d_model)
        reg_out = self.dist_kan2(dist_feats)      # (B, 1)

        return cls_out, reg_out, A

    def attention_penalty(self, A):
        return self.att_pool.penalization_term(A)

In [ ]:
%%writefile src/visualize.py
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def plot_history(train_loss, val_loss, save_path):
    plt.figure(figsize=(10, 5))
    plt.plot(train_loss, label='Train')
    plt.plot(val_loss, label='Val')
    plt.title('Loss Curve')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.savefig(save_path)
    plt.close()

def plot_cam(sequence, scores, save_path):
    # Normalize
    scores = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)
    plt.figure(figsize=(15, 3))
    plt.bar(range(len(scores)), scores, color='red', alpha=0.6)
    plt.xlabel('Position (pooled)')
    plt.ylabel('Importance')
    plt.title(f"CAM Feature Importance (Seq Len: {len(sequence)}, CAM Res: {len(scores)})")
    plt.savefig(save_path)
    plt.close()

In [ ]:
%%writefile src/interpretation.py
import torch

class Interpreter:
    def __init__(self, model):
        self.model = model
        self.grads = None
        self.acts = None

    def get_cam(self, seq, epi):
        self.model.eval()
        
        # Hook into Sequence CNN
        target_layer = self.model.seq_cnn[4] # The second Conv1d
        
        def fwd_hook(m, i, o): self.acts = o
        def bwd_hook(m, gi, go): self.grads = go[0]
        
        h1 = target_layer.register_forward_hook(fwd_hook)
        h2 = target_layer.register_full_backward_hook(bwd_hook)
        
        # Inference
        out, _, _ = self.model(seq.unsqueeze(0), epi.unsqueeze(0))
        
        # Backward
        self.model.zero_grad()
        out.backward()
        
        # CAM Calculation
        weights = torch.mean(self.grads, dim=2)[0] # Global pool over length
        cam = torch.zeros(self.acts.shape[2]).to(self.acts.device)
        
        for i, w in enumerate(weights):
            cam += w * self.acts[0, i, :]
            
        cam = torch.relu(cam) # ReLU
        
        h1.remove()
        h2.remove()
        return cam.detach().cpu().numpy()

In [ ]:
%%writefile train.py
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Subset
import yaml
import os
import glob
import numpy as np
import pickle
from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score

from src.epi_data_pipeline import EPIGenomicDataset
from src.dataset import EPIDataset
from src.model import Kansformer
from src.encoding import POCD_ND_Encoder
from src.visualize import plot_history

# 1. Setup
with open('configs/config.yaml') as f: config = yaml.safe_load(f)
os.makedirs(config['paths']['save_dir'], exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# 2. Data Loading (BEFORE encoder fitting — need real sequences)
bengi_dir = config['paths'].get('bengi_dir', './data/BENGI')
feats_config = config['paths'].get('feats_config', '')
ref_genome = config['paths'].get('ref_genome', '')

bengi_files = sorted(
    glob.glob(os.path.join(bengi_dir, '*.tsv.gz')) +
    glob.glob(os.path.join(bengi_dir, '*.tsv'))
)

USE_REAL_DATA = len(bengi_files) > 0 and os.path.exists(feats_config)

if USE_REAL_DATA:
    print(f"\n=== REAL DATA MODE ===")
    print(f"BENGI files: {len(bengi_files)}")
    for f_path in bengi_files:
        print(f"  {os.path.basename(f_path)}")

    genomic_ds = EPIGenomicDataset(
        bengi_paths=bengi_files,
        feats_config_path=feats_config,
        feats_order=config['data'].get('feats_order', None),
        seq_len=config['data'].get('seq_len_bp', 2_500_000),
        bin_size=config['data'].get('bin_size', 500),
        enhancer_window=config['data'].get('enhancer_window', 3000),
        promoter_window=config['data'].get('promoter_window', 3000),
        ref_genome_path=ref_genome if ref_genome else None,
    )

    # Split by chromosome
    valid_chroms = config['training'].get('valid_chroms', ['chr11', 'chr17'])
    test_chroms = config['training'].get('test_chroms', ['chr1', 'chr2'])
    chroms = genomic_ds.get_chrom_groups()
    labels = genomic_ds.get_labels()

    train_idx = [i for i, c in enumerate(chroms) if c not in valid_chroms and c not in test_chroms]
    val_idx = [i for i, c in enumerate(chroms) if c in valid_chroms]
    test_idx = [i for i, c in enumerate(chroms) if c in test_chroms]

    if len(val_idx) == 0:
        n = len(genomic_ds); train_size = int(0.8 * n)
        perm = np.random.permutation(n)
        train_idx = perm[:train_size].tolist()
        val_idx = perm[train_size:].tolist()

    # --- Fit POCD-ND encoder on REAL training sequences ---
    print("\nFitting POCD-ND Encoder on real training sequences...")
    encoder = POCD_ND_Encoder(k=config['data']['kmer_size'])

    train_labels_arr = np.array([labels[i] for i in train_idx])
    pos_train_idx = [train_idx[j] for j in np.where(train_labels_arr == 1)[0]]
    neg_train_idx = [train_idx[j] for j in np.where(train_labels_arr == 0)[0]]

    # Sample up to N sequences from each class for fitting
    MAX_FIT_SAMPLES = config['data'].get('encoder_fit_samples', 5000)
    np.random.seed(42)
    pos_sample = np.random.choice(pos_train_idx, size=min(MAX_FIT_SAMPLES, len(pos_train_idx)), replace=False)
    neg_sample = np.random.choice(neg_train_idx, size=min(MAX_FIT_SAMPLES, len(neg_train_idx)), replace=False)

    print(f"  Extracting {len(pos_sample)} positive + {len(neg_sample)} negative sequences...")
    pos_seqs = []
    for i in pos_sample:
        raw = genomic_ds[i]
        pos_seqs.append(raw["enhancer_seq"] + raw["promoter_seq"])
    neg_seqs = []
    for i in neg_sample:
        raw = genomic_ds[i]
        neg_seqs.append(raw["enhancer_seq"] + raw["promoter_seq"])

    encoder.fit(pos_seqs, neg_seqs, config['data']['sequence_length'])
    print(f"  Encoder fitted on {len(pos_seqs)} pos + {len(neg_seqs)} neg real sequences.")

    with open(f"{config['paths']['save_dir']}/encoder.pkl", 'wb') as f:
        pickle.dump(encoder, f)
    print("  Encoder saved.")

    # Wrap with POCD-ND encoding
    dataset = EPIDataset(config, encoder, source_dataset=genomic_ds)

    train_set = Subset(dataset, train_idx)
    val_set = Subset(dataset, val_idx)
    test_set = Subset(dataset, test_idx) if len(test_idx) > 0 else None

    # Compute class balance
    train_labels = [labels[i] for i in train_idx]
    n_pos = sum(train_labels)
    n_neg = len(train_labels) - n_pos
    pos_weight_val = n_neg / max(n_pos, 1)
    print(f"\nTrain: {len(train_set)}, Val: {len(val_set)}, Test: {len(test_idx)} "
          f"(val chroms: {valid_chroms}, test chroms: {test_chroms})")
    print(f"Class balance — pos: {n_pos} ({100*n_pos/len(train_labels):.1f}%), "
          f"neg: {n_neg} ({100*n_neg/len(train_labels):.1f}%), "
          f"pos_weight: {pos_weight_val:.2f}")

else:
    print(f"\n=== SYNTHETIC DATA MODE ===")
    print(f"BENGI dir '{bengi_dir}' not found or feats_config missing.")
    print(f"Using synthetic data for pipeline testing.")
    encoder = POCD_ND_Encoder(k=config['data']['kmer_size'])
    np.random.seed(42)
    dummy = ["".join(np.random.choice(['A','C','G','T'], config['data']['sequence_length'])) for _ in range(50)]
    encoder.fit(dummy, dummy, config['data']['sequence_length'])
    dataset = EPIDataset(config, encoder, num_synthetic=200)
    train_size = int(0.8 * len(dataset))
    train_set, val_set = random_split(dataset, [train_size, len(dataset)-train_size])
    test_set = None
    pos_weight_val = 1.0
    labels = None
    train_idx = list(range(train_size))

num_workers = config['training'].get('num_workers', 0)

# NO WeightedRandomSampler — reference KansformerEPI uses plain shuffle + BCELoss
train_loader = DataLoader(train_set, batch_size=config['data']['batch_size'],
                          shuffle=True, num_workers=num_workers)
print("Using shuffle (no weighted sampler — matches reference KansformerEPI)")

val_loader = DataLoader(val_set, batch_size=config['data']['batch_size'],
                        num_workers=num_workers)
if test_set is not None:
    test_loader = DataLoader(test_set, batch_size=config['data']['batch_size'],
                             num_workers=num_workers)

# 3. Model & Optimizer
model = Kansformer(config).to(device)
optimizer = optim.Adam(model.parameters(), lr=config['training']['lr'],
                       weight_decay=config['training'].get('weight_decay', 1e-4))

# Cosine annealing with warm restarts (reference uses T_0=5, T_mult=2)
use_cosine = config['training'].get('use_cosine_scheduler', True)
if use_cosine:
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=5, T_mult=2)
    print("Using CosineAnnealingWarmRestarts scheduler (T_0=5, T_mult=2)")
else:
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)
    print("Using ReduceLROnPlateau scheduler")

# BCELoss (matches reference — no logits, model outputs raw logits so we use BCEWithLogitsLoss)
crit_cls = nn.BCEWithLogitsLoss()
print("Using BCEWithLogitsLoss (no pos_weight — matches reference)")

crit_reg = nn.MSELoss()

total_params = sum(p.numel() for p in model.parameters())
train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModel: {total_params:,} params ({train_params:,} trainable)")

# 4. Training Loop
train_loss_hist, val_loss_hist = [], []
best_metric = -float('inf')  # Early stop on AUC + AUPR (like reference)
patience_counter = 0

use_augment = config.get('augmentation', {}).get('enabled', False)
print(f"\nStarting Training for {config['training']['epochs']} epochs...")
print(f"Augmentation: {'ON' if use_augment else 'OFF'}")
print(f"Early stopping: patience={config['training'].get('patience', 10)}, monitor=AUC+AUPR")
for epoch in range(config['training']['epochs']):
    model.train()
    dataset.augment = use_augment
    epoch_loss = 0
    train_preds_all, train_labels_all = [], []
    
    for batch in train_loader:
        seq = batch['seq'].float().to(device)
        epi = batch['epi'].float().to(device)
        lbl = batch['label'].float().to(device)
        dst = batch['dist'].float().to(device)
        enh_idx = batch['enh_idx'].float().to(device)
        prom_idx = batch['prom_idx'].float().to(device)
        
        optimizer.zero_grad()
        p_cls, p_reg, A = model(seq, epi, enh_idx, prom_idx)
        
        loss = crit_cls(p_cls, lbl) + (config['training']['lambda_dist'] * crit_reg(p_reg, dst))
        att_penalty = model.attention_penalty(A)
        loss = loss + config['training'].get('att_penalty_coeff', 0.1) * att_penalty
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), config['training'].get('grad_clip', 1.0))
        optimizer.step()
        epoch_loss += loss.item()
        
        # Collect train predictions for AUC/AUPR
        with torch.no_grad():
            train_preds_all.extend(torch.sigmoid(p_cls).cpu().numpy().flatten())
            train_labels_all.extend(lbl.cpu().numpy().flatten())

    # Step cosine scheduler per epoch
    if use_cosine:
        scheduler.step(epoch)
    
    # Validation
    model.eval()
    dataset.augment = False
    val_loss = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            seq = batch['seq'].float().to(device)
            epi = batch['epi'].float().to(device)
            lbl = batch['label'].float().to(device)
            dst = batch['dist'].float().to(device)
            enh_idx = batch['enh_idx'].float().to(device)
            prom_idx = batch['prom_idx'].float().to(device)
            p_cls, p_reg, _ = model(seq, epi, enh_idx, prom_idx)
            val_loss += (crit_cls(p_cls, lbl) + config['training']['lambda_dist'] * crit_reg(p_reg, dst)).item()
            all_preds.extend(torch.sigmoid(p_cls).cpu().numpy().flatten())
            all_labels.extend(lbl.cpu().numpy().flatten())
            
    avg_train = epoch_loss / max(len(train_loader), 1)
    avg_val = val_loss / max(len(val_loader), 1)
    train_loss_hist.append(avg_train)
    val_loss_hist.append(avg_val)
    
    preds_np = np.array(all_preds)
    labels_np = np.array(all_labels)
    acc = accuracy_score(labels_np, (preds_np > 0.5).astype(int))
    try:
        auc = roc_auc_score(labels_np, preds_np)
    except ValueError:
        auc = 0.0
    try:
        aupr = average_precision_score(labels_np, preds_np)
    except ValueError:
        aupr = 0.0
    
    # Compute train AUC/AUPR
    train_preds_np = np.array(train_preds_all)
    train_labels_np = np.array(train_labels_all)
    try:
        train_auc = roc_auc_score(train_labels_np, train_preds_np)
    except ValueError:
        train_auc = 0.0
    try:
        train_aupr = average_precision_score(train_labels_np, train_preds_np)
    except ValueError:
        train_aupr = 0.0
    
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1}/{config['training']['epochs']} | "
          f"train({train_auc:.4f}/{train_aupr:.4f})/vald({auc:.4f}/{aupr:.4f}): "
          f"TrainLoss: {avg_train:.4f} | ValLoss: {avg_val:.4f} | ValAcc: {acc:.4f}")

    if not use_cosine:
        scheduler.step(auc + aupr)
    
    # Early stopping on AUC + AUPR (matches reference)
    current_metric = auc + aupr
    if current_metric > best_metric:
        best_metric = current_metric
        torch.save(model.state_dict(), f"{config['paths']['save_dir']}/model_best.pth")
        patience_counter = 0
        print(f"  -> New best (AUC+AUPR={current_metric:.4f}), checkpoint saved.")
    else:
        patience_counter += 1
    
    if patience_counter >= config['training'].get('patience', 10):
        print(f"Early stopping at epoch {epoch+1}")
        break

# 5. Save & Final Evaluation
torch.save(model.state_dict(), f"{config['paths']['save_dir']}/model_final.pth")
plot_history(train_loss_hist, val_loss_hist, f"{config['paths']['save_dir']}/loss.png")

# --- Evaluate on validation set with best model ---
print("\n=== Final Validation Evaluation (best checkpoint) ===")
model.load_state_dict(torch.load(f"{config['paths']['save_dir']}/model_best.pth",
                                  map_location=device, weights_only=True))
model.eval()
dataset.augment = False
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in val_loader:
        seq = batch['seq'].float().to(device)
        epi = batch['epi'].float().to(device)
        lbl = batch['label'].float().to(device)
        enh_idx = batch['enh_idx'].float().to(device)
        prom_idx = batch['prom_idx'].float().to(device)
        p_cls, _, _ = model(seq, epi, enh_idx, prom_idx)
        all_preds.extend(torch.sigmoid(p_cls).cpu().numpy().flatten())
        all_labels.extend(lbl.cpu().numpy().flatten())
preds_np = np.array(all_preds)
labels_np = np.array(all_labels)
val_acc = accuracy_score(labels_np, (preds_np > 0.5).astype(int))
val_auc = roc_auc_score(labels_np, preds_np)
val_aupr = average_precision_score(labels_np, preds_np)
print(f"Val Accuracy: {val_acc:.4f}")
print(f"Val AUROC:    {val_auc:.4f}")
print(f"Val AUPR:     {val_aupr:.4f}")

# --- Evaluate on test set (if available) ---
if test_set is not None and len(test_idx) > 0:
    print("\n=== Test Evaluation (best checkpoint) ===")
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in test_loader:
            seq = batch['seq'].float().to(device)
            epi = batch['epi'].float().to(device)
            lbl = batch['label'].float().to(device)
            enh_idx = batch['enh_idx'].float().to(device)
            prom_idx = batch['prom_idx'].float().to(device)
            p_cls, _, _ = model(seq, epi, enh_idx, prom_idx)
            all_preds.extend(torch.sigmoid(p_cls).cpu().numpy().flatten())
            all_labels.extend(lbl.cpu().numpy().flatten())
    preds_np = np.array(all_preds)
    labels_np = np.array(all_labels)
    test_acc = accuracy_score(labels_np, (preds_np > 0.5).astype(int))
    test_auc = roc_auc_score(labels_np, preds_np)
    test_aupr = average_precision_score(labels_np, preds_np)
    print(f"Test Accuracy: {test_acc:.4f}")
    print(f"Test AUROC:    {test_auc:.4f}")
    print(f"Test AUPR:     {test_aupr:.4f}")

print("\nTraining Complete.")

In [ ]:
%%writefile evaluate.py
"""
Standalone evaluation script for POCD-KansformerEPI.
Loads the best checkpoint and evaluates on validation and test chromosomes.

Usage:
    python evaluate.py                       # Uses configs/config.yaml
    python evaluate.py --config path/to/config.yaml
    python evaluate.py --checkpoint path/to/model.pth
"""
import torch
import yaml
import numpy as np
import os
import glob
import pickle
import argparse
from sklearn.metrics import (accuracy_score, roc_auc_score,
                             average_precision_score, f1_score,
                             precision_score, recall_score,
                             confusion_matrix, classification_report)
from torch.utils.data import DataLoader, Subset

from src.epi_data_pipeline import EPIGenomicDataset
from src.dataset import EPIDataset
from src.model import Kansformer
from src.encoding import POCD_ND_Encoder


def evaluate_split(model, loader, device, split_name="Val"):
    """Evaluate model on a DataLoader split and print metrics."""
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            seq = batch['seq'].float().to(device)
            epi = batch['epi'].float().to(device)
            lbl = batch['label'].float().to(device)
            enh_idx = batch['enh_idx'].float().to(device)
            prom_idx = batch['prom_idx'].float().to(device)
            p_cls, _, _ = model(seq, epi, enh_idx, prom_idx)
            all_preds.extend(torch.sigmoid(p_cls).cpu().numpy().flatten())
            all_labels.extend(lbl.cpu().numpy().flatten())

    preds_np = np.array(all_preds)
    labels_np = np.array(all_labels)
    binary_preds = (preds_np > 0.5).astype(int)

    acc = accuracy_score(labels_np, binary_preds)
    try:
        auc = roc_auc_score(labels_np, preds_np)
    except ValueError:
        auc = float('nan')
    try:
        aupr = average_precision_score(labels_np, preds_np)
    except ValueError:
        aupr = float('nan')
    f1 = f1_score(labels_np, binary_preds, zero_division=0)
    prec = precision_score(labels_np, binary_preds, zero_division=0)
    rec = recall_score(labels_np, binary_preds, zero_division=0)
    cm = confusion_matrix(labels_np, binary_preds)

    print(f"\n{'='*50}")
    print(f"  {split_name} Results  ({len(labels_np)} samples)")
    print(f"{'='*50}")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  AUROC:     {auc:.4f}")
    print(f"  AUPR:      {aupr:.4f}")
    print(f"  F1 Score:  {f1:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}")
    print(f"\n  Confusion Matrix:")
    print(f"    TN={cm[0,0]}  FP={cm[0,1]}")
    print(f"    FN={cm[1,0]}  TP={cm[1,1]}")
    print(f"\n  Pos: {int(labels_np.sum())} ({100*labels_np.mean():.1f}%)  "
          f"Neg: {int(len(labels_np)-labels_np.sum())} ({100*(1-labels_np.mean()):.1f}%)")
    print(f"  Predicted Pos: {int(binary_preds.sum())}  "
          f"Predicted Neg: {int(len(binary_preds)-binary_preds.sum())}")
    print(f"{'='*50}")

    return {
        'accuracy': acc, 'auroc': auc, 'aupr': aupr,
        'f1': f1, 'precision': prec, 'recall': rec,
        'confusion_matrix': cm,
        'predictions': preds_np, 'labels': labels_np
    }


def main():
    parser = argparse.ArgumentParser(description="Evaluate POCD-KansformerEPI")
    parser.add_argument('--config', default='configs/config.yaml', help='Path to config file')
    parser.add_argument('--checkpoint', default=None, help='Path to model checkpoint (default: best)')
    parser.add_argument('--device', default=None, help='Device (cuda/cpu)')
    args = parser.parse_args()

    with open(args.config) as f:
        config = yaml.safe_load(f)

    device = torch.device(args.device if args.device else
                          ('cuda' if torch.cuda.is_available() else 'cpu'))
    print(f"Device: {device}")
    print(f"Config: {args.config}")

    # Load encoder
    encoder_path = f"{config['paths']['save_dir']}/encoder.pkl"
    if not os.path.exists(encoder_path):
        print(f"ERROR: Encoder not found at {encoder_path}")
        print("Run train.py first to fit and save the encoder.")
        return
    with open(encoder_path, 'rb') as f:
        encoder = pickle.load(f)
    print(f"Loaded encoder from {encoder_path}")

    # Load model
    model = Kansformer(config).to(device)
    ckpt_path = args.checkpoint
    if ckpt_path is None:
        ckpt_path = f"{config['paths']['save_dir']}/model_best.pth"
        if not os.path.exists(ckpt_path):
            ckpt_path = f"{config['paths']['save_dir']}/model_final.pth"
    model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
    print(f"Loaded model from {ckpt_path}")

    total_params = sum(p.numel() for p in model.parameters())
    print(f"Model params: {total_params:,}")

    # Load data
    bengi_dir = config['paths'].get('bengi_dir', './data/BENGI')
    feats_config = config['paths'].get('feats_config', '')
    ref_genome = config['paths'].get('ref_genome', '')

    bengi_files = sorted(
        glob.glob(os.path.join(bengi_dir, '*.tsv.gz')) +
        glob.glob(os.path.join(bengi_dir, '*.tsv'))
    )

    if len(bengi_files) == 0 or not os.path.exists(feats_config):
        print("ERROR: BENGI data or feats_config not found.")
        return

    genomic_ds = EPIGenomicDataset(
        bengi_paths=bengi_files,
        feats_config_path=feats_config,
        feats_order=config['data'].get('feats_order', None),
        seq_len=config['data'].get('seq_len_bp', 2_500_000),
        bin_size=config['data'].get('bin_size', 500),
        enhancer_window=config['data'].get('enhancer_window', 3000),
        promoter_window=config['data'].get('promoter_window', 3000),
        ref_genome_path=ref_genome if ref_genome else None,
    )

    dataset = EPIDataset(config, encoder, source_dataset=genomic_ds)
    dataset.augment = False  # No augmentation during evaluation

    chroms = genomic_ds.get_chrom_groups()
    labels = genomic_ds.get_labels()
    valid_chroms = config['training'].get('valid_chroms', ['chr11', 'chr17'])
    test_chroms = config['training'].get('test_chroms', ['chr1', 'chr2'])

    val_idx = [i for i, c in enumerate(chroms) if c in valid_chroms]
    test_idx = [i for i, c in enumerate(chroms) if c in test_chroms]
    train_idx = [i for i, c in enumerate(chroms) if c not in valid_chroms and c not in test_chroms]

    print(f"\nTrain: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")
    print(f"Val chroms: {valid_chroms}, Test chroms: {test_chroms}")

    num_workers = config['training'].get('num_workers', 0)
    batch_size = config['data']['batch_size']

    results = {}

    # Validate
    if len(val_idx) > 0:
        val_set = Subset(dataset, val_idx)
        val_loader = DataLoader(val_set, batch_size=batch_size, num_workers=num_workers)
        results['val'] = evaluate_split(model, val_loader, device, "Validation")

    # Test
    if len(test_idx) > 0:
        test_set = Subset(dataset, test_idx)
        test_loader = DataLoader(test_set, batch_size=batch_size, num_workers=num_workers)
        results['test'] = evaluate_split(model, test_loader, device, "Test")

    # Save results
    results_path = f"{config['paths']['save_dir']}/eval_results.npz"
    save_dict = {}
    for split_name, r in results.items():
        save_dict[f'{split_name}_predictions'] = r['predictions']
        save_dict[f'{split_name}_labels'] = r['labels']
        save_dict[f'{split_name}_auroc'] = r['auroc']
        save_dict[f'{split_name}_aupr'] = r['aupr']
    np.savez(results_path, **save_dict)
    print(f"\nResults saved to {results_path}")
    print("\nEvaluation Complete.")


if __name__ == '__main__':
    main()

## Verify Written Files

In [ ]:
import os

expected_files = [
    'configs/config.yaml',
    'src/__init__.py',
    'src/model_layers.py',
    'src/encoding.py',
    'src/epi_data_pipeline.py',
    'src/dataset.py',
    'src/model.py',
    'src/visualize.py',
    'src/interpretation.py',
    'train.py',
    'evaluate.py',
]

print('=== File Verification ===')
all_ok = True
for f in expected_files:
    exists = os.path.isfile(f)
    size = os.path.getsize(f) if exists else 0
    status = '✓' if exists and size > 0 else '✗ MISSING'
    print(f'  {status}  {f} ({size:,} bytes)')
    if not exists or size == 0:
        all_ok = False

if all_ok:
    print('\n✓ All files verified successfully!')
else:
    print('\n✗ Some files are missing — re-run the %%writefile cells above')


## Quick Model Sanity Check

Verify the model builds correctly and count parameters.

In [ ]:
import yaml, torch, sys
sys.path.insert(0, '.')

with open('configs/config.yaml') as f:
    config = yaml.safe_load(f)

from src.model import Kansformer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model = Kansformer(config).to(device)

total_params = sum(p.numel() for p in model.parameters())
train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {train_params:,}')

# Quick forward pass with dummy data
B = 2
seq = torch.randn(B, 64, config['data']['sequence_length'] - config['data']['kmer_size'] + 1).to(device)
epi = torch.randn(B, config['data']['n_epigenetic_features'], config['data']['epigenetic_bins']).to(device)
enh_idx = torch.tensor([[2500.0], [2500.0]]).to(device)
prom_idx = torch.tensor([[2500.0], [2500.0]]).to(device)

with torch.no_grad():
    cls_out, reg_out, A = model(seq, epi, enh_idx, prom_idx)

print(f'\ncls_out shape: {cls_out.shape}  (expected: [2, 1])')
print(f'reg_out shape: {reg_out.shape}  (expected: [2, 1])')
print(f'A shape:       {A.shape}  (expected: [2, 32, 628])')
print(f'\n✓ Model forward pass successful!')

# Architecture summary
print(f'\n=== v6 Architecture Summary ===')
print(f'Seq branch:  64ch → CNN(128→180) → AdaptivePool({config["model"]["n_tokens"]}) → BiLSTM → {config["model"]["n_tokens"]} tokens')
print(f'Epi branch:  {config["data"]["n_epigenetic_features"]}ch → CNN(180) → MaxPool(10) → BiLSTM → 500 tokens')
print(f'Total tokens: {config["model"]["n_tokens"]} + 500 = {config["model"]["n_tokens"] + 500}')
print(f'Transformer:  depth={config["model"]["num_layers"]}, heads={config["model"]["num_heads"]}, d_model={config["model"]["hidden_dim"]}')
print(f'FC head:      720 → 128 → KAN(64) → KAN(1)')
print(f'Pooling:      [enh_feat, prom_feat, attn_mean, attn_max] = 4×{config["model"]["hidden_dim"]} = {4*config["model"]["hidden_dim"]}')

del model, seq, epi, enh_idx, prom_idx, cls_out, reg_out, A
torch.cuda.empty_cache() if torch.cuda.is_available() else None


## Launch v6 Training

**Important:** Run training from the **terminal**, not the notebook, to avoid timeouts.

Training will run for up to 300 epochs with early stopping (patience=10).
Expected training time: **12-24 hours** on g2-standard-16 (1× NVIDIA L4).

### Option A: Background training (recommended)


In [ ]:
# Launch training in background — logs to training_v6.log
!nohup python train.py > training_v6.log 2>&1 &
!echo "Training launched! PID: $(cat /proc/self/status | head -1)"
!sleep 2
!tail -5 training_v6.log 2>/dev/null || echo "Log file not yet created, wait a moment..."


## Monitor Training Progress

Re-run this cell periodically to check progress.

In [ ]:
# Show last 20 lines of training log
!tail -20 training_v6.log 2>/dev/null || echo "No log file found. Has training started?"


In [ ]:
# Check GPU utilization
!nvidia-smi


In [ ]:
# Check if training is still running
import subprocess
result = subprocess.run(['pgrep', '-f', 'python train.py'], capture_output=True, text=True)
if result.stdout.strip():
    print(f"✓ Training is running (PID: {result.stdout.strip()})")
else:
    print("✗ Training is NOT running — check training_v6.log for errors")
    print("\nLast 10 lines of log:")
    !tail -10 training_v6.log 2>/dev/null


## Parse Training Results

Run after training completes.

In [ ]:
# Extract best epoch metrics from training log
import re

log_file = 'training_v6.log'
if not os.path.exists(log_file):
    print("No training log found!")
else:
    with open(log_file) as f:
        lines = f.readlines()

    # Find all epoch lines
    epoch_pattern = re.compile(
        r'Epoch (\d+)/\d+ \| '
        r'train\(([\d.]+)/([\d.]+)\)/vald\(([\d.]+)/([\d.]+)\): '
        r'TrainLoss: ([\d.]+) \| ValLoss: ([\d.]+) \| ValAcc: ([\d.]+)'
    )

    best_val_auc = 0
    best_epoch_line = ""
    epochs_found = 0

    for line in lines:
        m = epoch_pattern.search(line)
        if m:
            epochs_found += 1
            val_auc = float(m.group(4))
            if val_auc > best_val_auc:
                best_val_auc = val_auc
                best_epoch_line = line.strip()

    print(f"Total epochs completed: {epochs_found}")
    print(f"\nBest epoch (by Val AUC):")
    print(f"  {best_epoch_line}")

    # Print final results
    print(f"\n=== Final Output ===")
    for line in lines[-30:]:
        line = line.strip()
        if line and not line.startswith('  Loading') and not line.startswith('  Extracting'):
            print(line)


## Full Evaluation

Run `evaluate.py` to get detailed metrics on validation and test sets.

In [ ]:
!python evaluate.py


## Compare with Reference KansformerEPI

| Metric | v4 | v5 | **v6** | Reference Target |
|--------|-----|-----|--------|-----------------|
| Val AUROC | 0.8045 | 0.7984 | **?** | **0.9164** |
| Val AUPR | 0.3777 | 0.3731 | **?** | **0.6709** |
| Val Accuracy | 0.8143 | 0.8935 | **?** | **0.9122** |
| Test AUROC | 0.8305 | 0.8245 | **?** | N/A |
| Test AUPR | 0.4772 | 0.4685 | **?** | N/A |
| Parameters | 3.48M | 2.60M | **?** | ~similar |
| Total Tokens | 256 | 256 | **628** | 500 |
| FC Head Dim | 360 | 360 | **720** | 720 |
| Epochs Trained | 19 | 12 | **?** | 244 |

Fill in `?` with v6 results after training.


## Visualize Training

In [ ]:
from IPython.display import Image, display
import os

loss_plot = './checkpoints/loss.png'
if os.path.exists(loss_plot):
    display(Image(filename=loss_plot))
    print("Loss curve displayed above")
else:
    print("Loss plot not found — training may not have completed yet")
